# IndiaStay Hotels — Exploratory Data Analysis (Step 2)

Now that the data is clean, we ask it questions. This is called
**EDA (Exploratory Data Analysis)** — looking at the data from
different angles, with numbers and charts, to find patterns before
we build dashboards or make recommendations.

We'll answer the four questions from our business objective:
1. What affects **cancellations**?
2. Where does **revenue** come from?
3. Is there a **seasonal** pattern in demand?
4. How do **customers** behave (channels, payments, group size)?

Each section below: ask a question → run code → look at the chart →
write down what it tells us in plain English.

## Step 0: Import tools

- `pandas` → work with the table
- `matplotlib.pyplot` → draw charts
- `seaborn` → makes charts look nicer with less code

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (8, 5)

## Step 1: Load the cleaned data

We use the file we saved at the end of the cleaning notebook.
`parse_dates=[...]` tells pandas to read those columns as real dates
right away.

In [ ]:
df = pd.read_csv(
    "indiastay_hotel_bookings_clean.csv",
    parse_dates=["Booking_Date", "Check_In", "Check_Out"]
)

print("Shape:", df.shape)
df.head()

## Section 1: Overall business snapshot

Before slicing the data any further, let's get the big-picture
numbers: how many bookings, how much revenue, what's the overall
cancellation rate.

Note: we only count `Total_Amount` from bookings that were **not**
cancelled — a cancelled booking didn't actually earn any revenue.

In [ ]:
total_bookings = len(df)
total_revenue = df.loc[df['Is_Cancelled'] == 0, 'Total_Amount'].sum()
cancellation_rate = df['Is_Cancelled'].mean() * 100
avg_booking_value = df['Total_Amount'].mean()
avg_lead_time = df['Lead_Time_Days'].mean()

print(f"Total bookings:        {total_bookings:,}")
print(f"Total revenue:         Rs {total_revenue:,.0f}")
print(f"Cancellation rate:     {cancellation_rate:.1f}%")
print(f"Average booking value: Rs {avg_booking_value:,.0f}")
print(f"Average lead time:     {avg_lead_time:.1f} days")

**What this tells us:** roughly 1 in every 5-6 bookings gets
cancelled (~18%), and on average guests book about 4 weeks ahead of
their stay. Now let's dig into *why* cancellations happen.

## Section 2: Cancellation Analysis

### 2a. Does Room Type affect cancellation rate?

We group by `Room_Type` and calculate what percentage of bookings in
each group were cancelled (`Is_Cancelled` is 1 for cancelled, 0 for
not — so the **average** of that column is exactly the cancellation
rate).

In [ ]:
cancel_by_room = (df.groupby('Room_Type')['Is_Cancelled'].mean() * 100).sort_values(ascending=False)
print(cancel_by_room.round(1))

cancel_by_room.plot(kind='bar', color='steelblue')
plt.title("Cancellation Rate by Room Type")
plt.ylabel("Cancellation Rate (%)")
plt.xlabel("Room Type")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

**Finding:** cancellation rates are close together across room types
(around 18%), with Deluxe slightly higher and Standard/Suite slightly
lower. There's no single room type driving cancellations — it's a
fairly consistent risk across the board.

### 2b. Does Booking Channel affect cancellation rate?

Same idea, grouped by how the booking was made.

In [ ]:
cancel_by_channel = (df.groupby('Booking_Channel')['Is_Cancelled'].mean() * 100).sort_values(ascending=False)
print(cancel_by_channel.round(1))

cancel_by_channel.plot(kind='bar', color='indianred')
plt.title("Cancellation Rate by Booking Channel")
plt.ylabel("Cancellation Rate (%)")
plt.xlabel("Booking Channel")
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.show()

**Finding:** OTA (MakeMyTrip) and Walk-in bookings run slightly
higher cancellation rates (~19%) than Website or OTA-Goibibo (~17-18%).
The gap is small, but it's a reasonable starting point for a
recommendation like *"review the cancellation policy for OTA and
walk-in bookings."*

### 2c. Does lead time (how early someone books) affect cancellation?

We compare the **average lead time** for cancelled vs. not-cancelled
bookings.

In [ ]:
lead_time_by_cancellation = df.groupby('Is_Cancelled')['Lead_Time_Days'].mean()
lead_time_by_cancellation.index = ['Not Cancelled', 'Cancelled']
print(lead_time_by_cancellation.round(1))

lead_time_by_cancellation.plot(kind='bar', color=['seagreen', 'salmon'])
plt.title("Average Lead Time: Cancelled vs Not Cancelled")
plt.ylabel("Average Lead Time (days)")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

**Finding:** in this dataset, lead time is almost identical for
cancelled and non-cancelled bookings (~28 days either way) — so
booking early or late doesn't meaningfully predict cancellation here.
Good to know: not every variable will show a strong pattern, and
reporting "no real difference" honestly is just as valuable as
finding one.

## Section 3: Revenue Analysis

### 3a. Which cities generate the most revenue?

We filter out cancelled bookings first (they generated no revenue),
then sum `Total_Amount` per city and look at the top 10.

In [ ]:
revenue_by_city = (
    df[df['Is_Cancelled'] == 0]
    .groupby('City')['Total_Amount']
    .sum()
    .sort_values(ascending=False)
    .head(10)
)
print(revenue_by_city.round(0))

revenue_by_city.plot(kind='barh', color='darkorange')
plt.title("Top 10 Cities by Revenue")
plt.xlabel("Total Revenue (Rs)")
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

**Finding:** revenue is fairly evenly spread across IndiaStay's
top cities (Rishikesh, Delhi, Jaipur, Goa, Mumbai all within a similar
range), meaning the business isn't overly dependent on any single
city — a sign of a well-diversified portfolio.

### 3b. Which room type brings in the most total revenue?

Note this is different from Room_Rate — a room can have a high price
but if few people book it, it won't lead to much *total* revenue.

In [ ]:
revenue_by_room = (
    df[df['Is_Cancelled'] == 0]
    .groupby('Room_Type')['Total_Amount']
    .sum()
    .sort_values(ascending=False)
)
print(revenue_by_room.round(0))

revenue_by_room.plot(kind='bar', color='mediumpurple')
plt.title("Total Revenue by Room Type")
plt.ylabel("Total Revenue (Rs)")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

**Finding:** Suites and Premium rooms bring in the most total revenue
despite likely being booked less often than Standard/Deluxe — their
higher price per night more than makes up for lower volume. This
supports a recommendation to **promote Suite/Premium upgrades**.

## Section 4: Seasonal Demand

### Are some months busier than others?

We count bookings per month, making sure the months are shown in
calendar order (not alphabetical, which is pandas' default).

In [ ]:
month_order = ["January", "February", "March", "April", "May", "June",
               "July", "August", "September", "October", "November", "December"]

bookings_by_month = df['Booking_Month'].value_counts().reindex(month_order)
print(bookings_by_month)

bookings_by_month.plot(kind='line', marker='o', color='teal')
plt.title("Bookings by Month")
plt.ylabel("Number of Bookings")
plt.xlabel("Month")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

**Finding:** bookings are noticeably higher from January to August
and drop off from September to December, with the lowest point in
September/November. This points to a **low season around
Sep-Nov** — a good window for IndiaStay to run promotions or
discounted packages to fill rooms.

## Section 5: Customer Behavior

### 5a. Which booking channel do customers use most?

In [ ]:
channel_counts = df['Booking_Channel'].value_counts()
print(channel_counts)

channel_counts.plot(kind='bar', color='cornflowerblue')
plt.title("Bookings by Channel")
plt.ylabel("Number of Bookings")
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.show()

**Finding:** bookings are spread fairly evenly across channels
(Travel Agent, Mobile App, Walk-in, OTAs, Corporate, Website all
close together), meaning no single channel dominates — IndiaStay's
customer base books through many different paths.

### 5b. How many guests are typically in one booking?

In [ ]:
avg_guests = df['Guests'].mean()
print(f"Average guests per booking: {avg_guests:.2f}")

df['Guests'].value_counts().sort_index().plot(kind='bar', color='goldenrod')
plt.title("Distribution of Guests per Booking")
plt.xlabel("Number of Guests")
plt.ylabel("Number of Bookings")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

**Finding:** the average booking has just over 2 guests, with 2-guest
bookings being the most common — consistent with couples/small
families being the core customer base.

## Section 6: Summary of Key Findings

Pulling everything together, here's what we'd tell the business:

1. **Cancellations (~18% overall)** are fairly consistent across room
   types and customer segments — there's no single "problem segment,"
   but OTA and walk-in bookings run slightly higher, worth reviewing
   deposit/cancellation policy for those channels.
2. **Revenue is well diversified** across cities, reducing risk from
   any single location underperforming.
3. **Suites and Premium rooms** punch above their weight on revenue —
   worth upselling/promoting to guests at booking time.
4. **Sep-Nov is a clear low season** — a strong candidate for targeted
   discounts, packages, or marketing pushes to smooth out demand
   across the year.
5. **No single channel dominates bookings**, so marketing spend and
   channel-partner relationships should stay spread across Website,
   OTAs, Travel Agents, and Corporate — not concentrated in one.

Next step: we'll take these same kinds of questions into **SQL**, to
practice writing them as queries instead of pandas code.